# Experiment 2B — Token Independence Check

Compare token checker masks (from 5A) with universal circuit masks to measure what fraction of each circuit is **token-driven** vs **concept-driven**.

For every object that has both a universal circuit (Module 2) and a token checker mask (5A),
we compute at each layer:
- **concept_fraction** = |A \ B| / |A| — neurons in the universal circuit that are NOT in the token checker (concept-only)
- **token_fraction** = |A ∩ B| / |A| — neurons in the universal circuit that ARE in the token checker (possibly token-driven)

where A = universal circuit mask and B = token checker mask.

In [ ]:
import subprocess, sys, os, shutil
for pkg in ["h5py", "numpy", "pandas"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import numpy as np
import pandas as pd
import h5py
from pathlib import Path

N_LAYERS = 8

In [ ]:
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/CSP-Atlas")
else:
    DATA_DIR = Path("/Users/piotrwilam/Data/CSP-Atlas")

print(f"DATA_DIR: {DATA_DIR}")
assert DATA_DIR.exists(), f"DATA_DIR does not exist: {DATA_DIR}"

In [ ]:
UNIVERSAL_FILE = "universal_106x50.h5"
UNIVERSAL_PATH = DATA_DIR / UNIVERSAL_FILE

universal = {}
obj_types = {}

with h5py.File(UNIVERSAL_PATH, "r") as f:
    um = f["universal_masks"]
    for layer_id in range(N_LAYERS):
        layer_key = f"layer_{layer_id}"
        if layer_key not in um:
            continue
        for ds_name in um[layer_key]:
            if ds_name not in universal:
                universal[ds_name] = {}
                obj_types[ds_name] = ds_name.split("__", 1)[0] if "__" in ds_name else ds_name.split("_", 1)[0]
            universal[ds_name][layer_id] = np.array(um[layer_key][ds_name], dtype=bool)

all_objects = sorted(universal.keys())
print(f"Loaded {len(all_objects)} universal circuits")

In [ ]:
TOKEN_CHECKER_FILE = "token_checker_masks.h5"
TOKEN_CHECKER_PATH = DATA_DIR / TOKEN_CHECKER_FILE

token_chk = {}

with h5py.File(TOKEN_CHECKER_PATH, "r") as f:
    tcm = f["token_checker_masks"]
    for layer_id in range(N_LAYERS):
        layer_key = f"layer_{layer_id}"
        if layer_key not in tcm:
            continue
        for ds_name in tcm[layer_key]:
            if ds_name not in token_chk:
                token_chk[ds_name] = {}
            token_chk[ds_name][layer_id] = np.array(tcm[layer_key][ds_name], dtype=bool)

print(f"Loaded {len(token_chk)} token checker masks")
print(f"Objects in both: {len(set(universal.keys()) & set(token_chk.keys()))}")

In [ ]:
KEYWORD_MAP = {
    # --- AST nodes with dedicated keywords ---
    "ast__Import":       {"keyword": "import",   "forbidden_nodes": ["Import", "ImportFrom"]},
    "ast__ImportFrom":   {"keyword": "from",     "forbidden_nodes": ["Import", "ImportFrom"]},
    "ast__Break":        {"keyword": "break",    "forbidden_nodes": ["Break"]},
    "ast__Pass":         {"keyword": "pass",     "forbidden_nodes": ["Pass"]},
    "ast__Continue":     {"keyword": "continue", "forbidden_nodes": ["Continue"]},
    "ast__Assert":       {"keyword": "assert",   "forbidden_nodes": ["Assert"]},
    "ast__If":           {"keyword": "if",       "forbidden_nodes": ["If", "IfExp"]},
    "ast__While":        {"keyword": "while",    "forbidden_nodes": ["While"]},
    "ast__For":          {"keyword": "for",      "forbidden_nodes": ["For", "AsyncFor", "ListComp", "SetComp", "DictComp", "GeneratorExp"]},
    "ast__Return":       {"keyword": "return",   "forbidden_nodes": ["Return"]},
    "ast__With":         {"keyword": "with",     "forbidden_nodes": ["With", "AsyncWith"]},
    "ast__Raise":        {"keyword": "raise",    "forbidden_nodes": ["Raise"]},
    "ast__Delete":       {"keyword": "del",      "forbidden_nodes": ["Delete"]},
    "ast__Global":       {"keyword": "global",   "forbidden_nodes": ["Global"]},
    "ast__Nonlocal":     {"keyword": "nonlocal", "forbidden_nodes": ["Nonlocal"]},
    "ast__Lambda":       {"keyword": "lambda",   "forbidden_nodes": ["Lambda"]},
    "ast__Yield":        {"keyword": "yield",    "forbidden_nodes": ["Yield", "YieldFrom"]},
    "ast__YieldFrom":    {"keyword": "yield",    "forbidden_nodes": ["Yield", "YieldFrom"]},
    "ast__Try":          {"keyword": "try",      "forbidden_nodes": ["Try"]},
    "ast__ClassDef":     {"keyword": "class",    "forbidden_nodes": ["ClassDef"]},
    "ast__FunctionDef":  {"keyword": "def",      "forbidden_nodes": ["FunctionDef", "AsyncFunctionDef"]},
    "ast__AsyncFor":     {"keyword": "async",    "forbidden_nodes": ["AsyncFor", "AsyncFunctionDef", "AsyncWith"]},
    "ast__AsyncFunctionDef": {"keyword": "async", "forbidden_nodes": ["AsyncFor", "AsyncFunctionDef", "AsyncWith"]},
    "ast__AsyncWith":    {"keyword": "async",    "forbidden_nodes": ["AsyncFor", "AsyncFunctionDef", "AsyncWith"]},

    # --- Builtins with short common-English tokens ---
    "builtin__list":     {"keyword": "list",     "forbidden_nodes": [],  "forbidden_names": ["list"]},
    "builtin__set":      {"keyword": "set",      "forbidden_nodes": [],  "forbidden_names": ["set"]},
    "builtin__map":      {"keyword": "map",      "forbidden_nodes": [],  "forbidden_names": ["map"]},
    "builtin__open":     {"keyword": "open",     "forbidden_nodes": [],  "forbidden_names": ["open"]},
    "builtin__type":     {"keyword": "type",     "forbidden_nodes": [],  "forbidden_names": ["type"]},
    "builtin__hash":     {"keyword": "hash",     "forbidden_nodes": [],  "forbidden_names": ["hash"]},
    "builtin__id":       {"keyword": "id",       "forbidden_nodes": [],  "forbidden_names": ["id"]},
    "builtin__all":      {"keyword": "all",      "forbidden_nodes": [],  "forbidden_names": ["all"]},
    "builtin__any":      {"keyword": "any",      "forbidden_nodes": [],  "forbidden_names": ["any"]},
    "builtin__sum":      {"keyword": "sum",      "forbidden_nodes": [],  "forbidden_names": ["sum"]},
    "builtin__min":      {"keyword": "min",      "forbidden_nodes": [],  "forbidden_names": ["min"]},
    "builtin__max":      {"keyword": "max",      "forbidden_nodes": [],  "forbidden_names": ["max"]},
    "builtin__next":     {"keyword": "next",     "forbidden_nodes": [],  "forbidden_names": ["next"]},
    "builtin__input":    {"keyword": "input",    "forbidden_nodes": [],  "forbidden_names": ["input"]},
    "builtin__len":      {"keyword": "len",      "forbidden_nodes": [],  "forbidden_names": ["len"]},
    "builtin__range":    {"keyword": "range",    "forbidden_nodes": [],  "forbidden_names": ["range"]},
    "builtin__filter":   {"keyword": "filter",   "forbidden_nodes": [],  "forbidden_names": ["filter"]},
    "builtin__print":    {"keyword": "print",    "forbidden_nodes": [],  "forbidden_names": ["print"]},
    "builtin__int":      {"keyword": "int",      "forbidden_nodes": [],  "forbidden_names": ["int"]},
    "builtin__float":    {"keyword": "float",    "forbidden_nodes": [],  "forbidden_names": ["float"]},
    "builtin__str":      {"keyword": "str",      "forbidden_nodes": [],  "forbidden_names": ["str"]},
    "builtin__bool":     {"keyword": "bool",     "forbidden_nodes": [],  "forbidden_names": ["bool"]},
    "builtin__round":    {"keyword": "round",    "forbidden_nodes": [],  "forbidden_names": ["round"]},
    "builtin__zip":      {"keyword": "zip",      "forbidden_nodes": [],  "forbidden_names": ["zip"]},
    "builtin__sorted":   {"keyword": "sorted",   "forbidden_nodes": [],  "forbidden_names": ["sorted"]},
    "builtin__super":    {"keyword": "super",    "forbidden_nodes": [],  "forbidden_names": ["super"]},
    "builtin__iter":     {"keyword": "iter",     "forbidden_nodes": [],  "forbidden_names": ["iter"]},
    "builtin__object":   {"keyword": "object",   "forbidden_nodes": [],  "forbidden_names": ["object"]},
    "builtin__bytes":    {"keyword": "bytes",    "forbidden_nodes": [],  "forbidden_names": ["bytes"]},
    "builtin__dict":     {"keyword": "dict",     "forbidden_nodes": [],  "forbidden_names": ["dict"]},
    "builtin__tuple":    {"keyword": "tuple",    "forbidden_nodes": [],  "forbidden_names": ["tuple"]},
    "builtin__property": {"keyword": "property", "forbidden_nodes": [],  "forbidden_names": ["property"]},
    "builtin__complex":  {"keyword": "complex",  "forbidden_nodes": [],  "forbidden_names": ["complex"]},
    "builtin__reversed": {"keyword": "reversed", "forbidden_nodes": [],  "forbidden_names": ["reversed"]},
}

print(f"KEYWORD_MAP: {len(KEYWORD_MAP)} objects with testable keywords")

In [ ]:
def compare_masks(universal_mask, token_mask):
    """
    Compare universal circuit (A) with token checker (B).

    Returns dict with:
        A_only: neurons in universal but not in token checker (concept-driven)
        A_and_B: neurons in both (possibly token-driven)
        B_only: neurons in token checker but not in universal (token-reactive, not concept)
        token_fraction: |A∩B| / |A| (fraction of universal that is token-explained)
        concept_fraction: |A\B| / |A| (fraction of universal that is concept-only)
    """
    A = universal_mask
    B = token_mask

    a_only = np.logical_and(A, ~B).sum()
    a_and_b = np.logical_and(A, B).sum()
    b_only = np.logical_and(~A, B).sum()
    a_size = A.sum()
    b_size = B.sum()

    token_fraction = float(a_and_b / a_size) if a_size > 0 else 0.0
    concept_fraction = float(a_only / a_size) if a_size > 0 else 0.0

    return {
        "A_size": int(a_size),
        "B_size": int(b_size),
        "A_only": int(a_only),
        "A_and_B": int(a_and_b),
        "B_only": int(b_only),
        "token_fraction": round(token_fraction, 4),
        "concept_fraction": round(concept_fraction, 4),
    }

In [ ]:
comparison_records = []

for obj_key in sorted(KEYWORD_MAP.keys()):
    if obj_key not in token_chk:
        continue  # no token checker for this object (insufficient prompts)
    if obj_key not in universal:
        continue  # no universal circuit for this object

    for layer_id in range(N_LAYERS):
        result = compare_masks(
            universal[obj_key][layer_id],
            token_chk[obj_key][layer_id]
        )
        result["object"] = obj_key
        result["type"] = "ast" if obj_key.startswith("ast__") else "builtin"
        result["keyword"] = KEYWORD_MAP[obj_key]["keyword"]
        result["layer"] = layer_id
        comparison_records.append(result)

comparison_df = pd.DataFrame(comparison_records)
print(f"Comparison rows: {len(comparison_df)}")
print(f"Objects compared: {comparison_df['object'].nunique()}")

In [ ]:
summary = comparison_df.groupby(["object", "type", "keyword"]).agg({
    "concept_fraction": "mean",
    "token_fraction": "mean",
    "A_size": "mean",
    "A_and_B": "mean",
    "A_only": "mean",
    "B_only": "mean",
}).round(4).sort_values("concept_fraction", ascending=False)

print("Per-object summary (sorted by concept fraction):")
display(summary)

In [ ]:
print(f"Full detail: {len(comparison_df)} rows")
display(comparison_df)

In [ ]:
no_keyword = [obj for obj in all_objects if obj not in KEYWORD_MAP]
print(f"Objects without testable keyword: {len(no_keyword)}")
print("These circuits have no keyword token confound — token independence is not applicable.")
for obj in no_keyword:
    print(f"  {obj}")

In [ ]:
print("=== TOKEN INDEPENDENCE RESULTS ===\n")

# Overall
mean_concept = comparison_df["concept_fraction"].mean()
mean_token = comparison_df["token_fraction"].mean()
print(f"Across all tested circuits and layers:")
print(f"  Mean concept-only fraction: {mean_concept:.3f}")
print(f"  Mean token-overlap fraction: {mean_token:.3f}")
print()

# By type
for obj_type in ["ast", "builtin"]:
    subset = comparison_df[comparison_df["type"] == obj_type]
    if len(subset) == 0:
        continue
    print(f"{obj_type.upper()} circuits:")
    print(f"  Mean concept fraction: {subset['concept_fraction'].mean():.3f}")
    print(f"  Mean token fraction:   {subset['token_fraction'].mean():.3f}")
    print()

# Highlight: modular circuits (the top scorers from Experiment 1)
MODULAR = ["ast__Import", "ast__Break", "ast__Pass", "ast__ImportFrom",
           "ast__Continue", "ast__Assert"]
mod_subset = comparison_df[comparison_df["object"].isin(MODULAR)]
if len(mod_subset) > 0:
    print("MODULAR circuits (score > 0 in Experiment 1):")
    print(f"  Mean concept fraction: {mod_subset['concept_fraction'].mean():.3f}")
    print(f"  Mean token fraction:   {mod_subset['token_fraction'].mean():.3f}")
    print()

# Any circuit where token_fraction > 0.5 at any layer?
high_token = comparison_df[comparison_df["token_fraction"] > 0.5]
if len(high_token) > 0:
    print(f"WARNING: {high_token['object'].nunique()} circuits have token_fraction > 0.5 at some layer:")
    for obj in high_token["object"].unique():
        layers = high_token[high_token["object"] == obj]["layer"].tolist()
        print(f"  {obj}: layers {layers}")

In [ ]:
comparison_df.to_csv(DATA_DIR / "token_independence_detail.csv", index=False)
summary.to_csv(DATA_DIR / "token_independence_summary.csv")
pd.DataFrame({"object": no_keyword, "status": "no_keyword"}).to_csv(
    DATA_DIR / "token_independence_no_keyword.csv", index=False
)

print("Saved:")
print(f"  {DATA_DIR / 'token_independence_detail.csv'}")
print(f"  {DATA_DIR / 'token_independence_summary.csv'}")
print(f"  {DATA_DIR / 'token_independence_no_keyword.csv'}")